In [200]:
import pandas as pd
import numpy as np
from sklearn.model_selection import KFold , train_test_split, GridSearchCV,ParameterGrid
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import (OrdinalEncoder , OneHotEncoder )
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer , TransformedTargetRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.linear_model import (LinearRegression , Ridge , Lasso)
from sklearn.ensemble import (RandomForestRegressor , GradientBoostingRegressor)
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

### 1.Importer et couper le jeux de données en train et test

In [201]:
df = pd.read_csv("data_cleaning/annonces.csv" , sep=";")
df.head()
print(df.duplicated().sum(), "lignes en double sur", len(df))
print(df.drop(columns=["Loyer_TTC"]).duplicated().sum(), "doublons sans compter le loyer")
df["station"].unique()
df["station"] = df["station"].replace({
    "Non/Non_renseigner":        "Aucun",
    "Garage":                    "Garage",
    "Garage fermé":              "Garage",
    "Garage, Garage fermé":      "Garage",
    "Garage, Sous-sol":          "Garage",
    "Garage, Parking":           "Garage",
    "Sous-sol":                  "Couvert/Sous-sol",
    "Stationnement en sous-sol": "Couvert/Sous-sol",
    "Couvert":                   "Couvert/Sous-sol",
    "Stationnement couvert":     "Couvert/Sous-sol",
    "Parking":                   "Extérieur",
    "Aérien":                    "Extérieur",
    "Stationnement aérien":      "Extérieur",
})
print(df["station"].value_counts())



4 lignes en double sur 310
6 doublons sans compter le loyer
station
Aucun               172
Extérieur            74
Garage               42
Couvert/Sous-sol     22
Name: count, dtype: int64


In [202]:
X=df.drop(columns=["Loyer_TTC"])
Y = df["Loyer_TTC"]
x_train , x_test , y_train , y_test = train_test_split(X,Y,test_size=0.25 ,random_state=42)
print(x_train.isna().sum())
print(y_train.isna().sum())

Surface      0
Charges      0
Pièces       0
label_eco    0
kWh          0
kgCO2        0
Quartiers    0
station      0
dtype: int64
0


In [203]:
#Imputer
set(df["label_eco"])



{'B+', 'C', 'D', 'E', 'F'}

Avec .ravel() :


x_train["label_eco"] = encoder.fit_transform(x_train[["label_eco"]]).ravel()

x_test["label_eco"]  = encoder.transform(x_test[["label_eco"]]).ravel()

Ou avec des doubles crochets des deux côtés, sans .ravel() :

x_train[["label_eco"]] = encoder.fit_transform(x_train[["label_eco"]])

x_test[["label_eco"]]  = encoder.transform(x_test[["label_eco"]])

In [204]:
df["label_eco"].unique()

<StringArray>
['E', 'D', 'C', 'B+', 'F']
Length: 5, dtype: str

In [205]:
def creer_pipeline(methode , avec_pca=False): #exemple methode = LogisticRegression()
    ordre = [['B+', 'C', 'D', 'E', 'F']]
    pipe_encoder = Pipeline(steps=[
        ("encoder" , OrdinalEncoder(categories=ordre))
    ])

    pipe_one_hot = Pipeline(steps=[
        ("encoder" , OneHotEncoder())
    ])
    pipe_pca = Pipeline(steps=[
        ("scale" , StandardScaler()),
        ("PCA" , PCA() )
    ])
    preprocesser = ColumnTransformer(transformers=[
        ("ord" , pipe_encoder ,["label_eco"] ) ,
        ("one_hot" , pipe_one_hot , ["Quartiers", "station"] )
    ], remainder="passthrough")
    preprocesser_avec_pca = ColumnTransformer(transformers=[
        ("ord" , pipe_encoder ,["label_eco"]) ,
        ("one_hot" , pipe_one_hot , ["Quartiers" , "station"]) ,
        ("pca" , pipe_pca ,["Surface" , "Charges" , "kWh" , "kgCO2"])
    ], remainder="passthrough")
    if avec_pca :
        return Pipeline(steps=[
            ("preprocesser" , preprocesser_avec_pca),
            ("modele" ,methode )
        ])
    return Pipeline(steps=[
               ("preprocesser" , preprocesser),
               ("modele" ,methode )
           ])
def avec_log(pipeline):
    return TransformedTargetRegressor(regressor=pipeline, func=np.log, inverse_func=np.exp)

In [206]:
# Grilles choisies selon la taille des données (~230 annonces d'entraînement) :
# - paramètres qui règlent la complexité, avec des valeurs prudentes (min_samples_leaf >= 2, arbres peu profonds)
# - échelle logarithmique pour les paramètres de régularisation (alpha)
# - learning_rate et n_estimators cherchés ensemble pour le gradient boosting
alphas = np.logspace(-4, 3, 15)   # élargie vers le bas : le meilleur alpha (log) était au bord, à 0.001

grilles = {
    "gradient boosting": (GradientBoostingRegressor(random_state=42),
                          {"n_estimators": [100, 200, 400],
                           "learning_rate": [0.01, 0.02,0.03],
                           "max_depth": [1,2, 3, 4,5],
                           "min_samples_leaf": list(range(1,10)),
                           "max_features": [0.5,1]}),          # 108 combinaisons
    "random forest regression": (RandomForestRegressor(n_estimators=200, random_state=42),
                                 {"max_depth": [4, 6, 8, None],
                                  "min_samples_leaf": [2, 3, 5, 10],
                                  "max_features": [0.3, 0.5, 1.0]}),  # 48 combinaisons
    "Régression Lasso": (Lasso(max_iter=10000), {"alpha": alphas}),
    "Régression Ridge": (Ridge(), {"alpha": alphas}),
}

def prefixer(grille, prefixe):
    return {prefixe + nom: valeurs for nom, valeurs in grille.items()}

models = {nom: (creer_pipeline(modele), prefixer(grille, "modele__"))
          for nom, (modele, grille) in grilles.items()}

In [207]:
from math import inf
cv = KFold(5)
metric ={"MSE" : "neg_mean_squared_error" ,
         "MAE" : "neg_mean_absolute_error" ,
         "R2" : "r2"
         }
meilleur = None
best_score = inf
best_parametre = None
best_estimateur = None
for nom_methode , (pipe_line , grid_param) in models.items() :
    print( f"{nom_methode}")
    grid_search = GridSearchCV(estimator=pipe_line , param_grid=grid_param,cv =cv ,
                               scoring=metric , refit ="MSE" , n_jobs=-1)
    grid_search.fit(x_train,y_train)
    i = grid_search.best_index_
    res = grid_search.cv_results_
    print("Best parametres :", grid_search.best_params_)
    print("MSE :", -res["mean_test_MSE"][i])
    print("MAE :", -res["mean_test_MAE"][i])
    print("R2  :",  res["mean_test_R2"][i])
    if -res["mean_test_MSE"][i] <=best_score :
        best_score = -res["mean_test_MSE"][i]
        meilleur = nom_methode 
        best_parametre =grid_search.best_params_
        best_estimateur = grid_search.best_estimator_
print(meilleur )
print(best_parametre)


gradient boosting
Best parametres : {'modele__learning_rate': 0.02, 'modele__max_depth': 5, 'modele__max_features': 0.5, 'modele__min_samples_leaf': 1, 'modele__n_estimators': 100}
MSE : 45860.67631315153
MAE : 128.9529271127409
R2  : 0.7481573122350789
random forest regression
Best parametres : {'modele__max_depth': None, 'modele__max_features': 0.5, 'modele__min_samples_leaf': 2}
MSE : 47767.39928428248
MAE : 120.58126356223815
R2  : 0.7655693748814857
Régression Lasso
Best parametres : {'modele__alpha': np.float64(10.0)}
MSE : 64446.70173123681
MAE : 137.39601137725396
R2  : 0.7120274617058207
Régression Ridge
Best parametres : {'modele__alpha': np.float64(31.622776601683793)}
MSE : 63326.75425034061
MAE : 138.14380557636113
R2  : 0.715730124251962
gradient boosting
{'modele__learning_rate': 0.02, 'modele__max_depth': 5, 'modele__max_features': 0.5, 'modele__min_samples_leaf': 1, 'modele__n_estimators': 100}


In [208]:
    # ou meilleurs["Random Forest"]

y_pred = best_estimateur.predict(x_test)
print("MAE  test :", mean_absolute_error(y_test, y_pred))
print("RMSE test :", np.sqrt(mean_squared_error(y_test, y_pred)))
print("R²   test :", r2_score(y_test, y_pred))
print("R²  train :", best_estimateur.score(x_train, y_train))
resultats = pd.DataFrame({"y_pred": y_pred, "y_reel": y_test})


erreurs = pd.DataFrame({"reel": y_test, "pred": y_pred})
erreurs["ecart"] = (erreurs["pred"] - erreurs["reel"]).abs()
display(erreurs.join(x_test).sort_values("ecart", ascending=False).head(10))


MAE  test : 99.12259709447747
RMSE test : 136.04096542261627
R²   test : 0.8518137996990238
R²  train : 0.9559567549525547


,reel,pred,ecart,Surface,Charges,Pièces,label_eco,kWh,kgCO2,Quartiers,station
302,1733.0,1211.611045,521.388955,94.0,275.0,4.0,D,143.0,30.0,Quartiers Nord-Ouest,Garage
179,1366.0,1025.898496,340.101504,86.0,16.0,4.0,C,135.0,29.0,Quartiers Centre,Aucun
182,1555.0,1215.178363,339.821637,93.0,85.0,4.0,D,172.0,34.0,Quartiers Centre,Aucun
6,459.0,792.710931,333.710931,24.0,140.0,2.0,C,89.0,18.0,Quartiers Centre,Aucun
196,1386.0,1069.958190,316.041810,84.0,120.0,3.0,B+,93.0,3.0,Quartiers Centre,Garage
198,1150.0,921.724748,228.275252,53.0,150.0,3.0,D,187.0,6.0,Quartiers Centre,Extérieur
158,995.0,786.514114,208.485886,52.0,75.0,2.0,C,75.0,12.0,Quartiers Centre,Extérieur
202,950.0,742.421154,207.578846,35.0,80.0,2.0,C,144.0,26.0,Quartiers Ouest,Aucun
197,1230.0,1023.178703,206.821297,63.0,80.0,4.0,D,240.0,42.0,Quartiers Centre,Aucun
114,799.0,997.492390,198.492390,70.0,87.0,4.0,E,297.0,48.0,Quartiers Nord-Ouest,Aucun


1. Les « fraudes » (grande surface, petit loyer)
Que ce soient des arnaques ou des colocations, la conclusion est la même : ces annonces ne sont pas fiables pour ton modèle, il faut les retirer avec une règle écrite dans data_cleaning.py. 
annonces « incohérentes (loyer au m² anormalement bas : possibles fraudes ou colocations) ». 

Ça reste prudent, sans affirmer que ce sont des fraudes, puisque tu ne peux pas le prouver.

2. Les loyers au-dessus de 1000 € : ce n'est pas maison contre appartement
Ton scraping ne récupère que des appartements : la recherche porte sur /louer/appartement/, et la ligne 77 ne garde que les liens /immobilier/location/appartement/. Il n'y a donc aucune maison dans tes données.

Les vraies causes, d'après les chiffres :

Peu d'exemples : seulement 50 annonces sur 306 (16 %) dépassent 1000 €, et elles vont de 1004 € à 4000 €. Le modèle a très peu d'exemples pour apprendre ce haut de marché, qui est très dispersé.


Des informations manquent : à surface égale, un appartement à 1500 € et un autre à 1000 € diffèrent souvent par le standing, le meublé, une terrasse, un parking, l'étage. Ton modèle ne voit rien de tout ça.

Prédire le log du loyer. C'est la solution la plus efficace pour ce problème. Les gros loyers pèsent moins dans l'erreur, et le modèle raisonne en pourcentages plutôt qu'en euros :

In [209]:
from sklearn.base import clone

def avec_log(pipeline):
    return TransformedTargetRegressor(regressor=pipeline, func=np.log, inverse_func=np.exp)

# Mêmes modèles et mêmes grilles que `models`, mais sur log(loyer)
models_log = {f"{nom} (log)": (avec_log(creer_pipeline(clone(modele))), prefixer(grille, "regressor__modele__"))
              for nom, (modele, grille) in grilles.items()}

In [210]:
meilleur = None
best_score = inf
best_parametre = None
best_estimateur = None
for nom_methode , (pipe_line , grid_param) in models_log.items() :
    print( f"{nom_methode}")
    grid_search = GridSearchCV(estimator=pipe_line , param_grid=grid_param,cv =cv ,
                               scoring=metric , refit ="MSE" , n_jobs=-1)
    grid_search.fit(x_train,y_train)
    i = grid_search.best_index_
    res = grid_search.cv_results_
    print("Best parametres :", grid_search.best_params_)
    print("MSE :", -res["mean_test_MSE"][i])
    print("MAE :", -res["mean_test_MAE"][i])
    print("R2  :",  res["mean_test_R2"][i])
    if -res["mean_test_MSE"][i] <=best_score :
        best_score = -res["mean_test_MSE"][i]
        meilleur = nom_methode 
        best_parametre =grid_search.best_params_
        best_estimateur = grid_search.best_estimator_


gradient boosting (log)
Best parametres : {'regressor__modele__learning_rate': 0.02, 'regressor__modele__max_depth': 2, 'regressor__modele__max_features': 0.5, 'regressor__modele__min_samples_leaf': 7, 'regressor__modele__n_estimators': 400}
MSE : 51976.809021833804
MAE : 125.42340342512298
R2  : 0.748183970538667
random forest regression (log)
Best parametres : {'regressor__modele__max_depth': 4, 'regressor__modele__max_features': 1.0, 'regressor__modele__min_samples_leaf': 2}
MSE : 50345.48912509462
MAE : 124.71754698164628
R2  : 0.755433728795874
Régression Lasso (log)
Best parametres : {'regressor__modele__alpha': np.float64(0.0001)}
MSE : 55747.024397199784
MAE : 125.92714958549261
R2  : 0.7616448043892241
Régression Ridge (log)
Best parametres : {'regressor__modele__alpha': np.float64(0.0001)}
MSE : 55657.63692929163
MAE : 125.95128574399503
R2  : 0.7618223604971878


In [211]:
print(meilleur )
print(best_parametre)
y_pred = best_estimateur.predict(x_test)
print("MAE  test :", mean_absolute_error(y_test, y_pred))
print("RMSE test :", np.sqrt(mean_squared_error(y_test, y_pred)))
print("R²   test :", r2_score(y_test, y_pred))
print("R²  train :", best_estimateur.score(x_train, y_train))
resultats = pd.DataFrame({"y_pred": y_pred, "y_reel": y_test})
print(resultats)

random forest regression (log)
{'regressor__modele__max_depth': 4, 'regressor__modele__max_features': 1.0, 'regressor__modele__min_samples_leaf': 2}
MAE  test : 96.72961651925253
RMSE test : 135.06578539173492
R²   test : 0.8539306662630867
R²  train : 0.9073740761665715
          y_pred  y_reel
289   457.496584   400.0
9     940.348134   997.0
57    828.683946   925.0
60    454.855258   425.0
25    586.869705   599.0
..           ...     ...
30    466.455956   477.0
22    467.835016   500.0
262  1013.952624  1200.0
56   1007.567540   911.0
148   468.436510   395.0

[78 rows x 2 columns]
